# Knowledge Extractor - Extracción de Entidades ATC

Este notebook procesa documentos PDF de fraseología aeronáutica para extraer entidades y relaciones usando NLP con Ollama.

## Importación de librerías

Importamos todas las dependencias necesarias para el procesamiento:
- `ollama`: Para interactuar con el modelo de lenguaje local
- `json`: Para manejar estructuras de datos JSON
- `re`: Para expresiones regulares
- `os`: Para operaciones de sistema de archivos
- `pypdf`: Para leer y extraer texto de archivos PDF

In [ ]:
import ollama
import json
import re
import os
from pypdf import PdfReader

import numpy as np
from sentence_transformers import SentenceTransformer
import nltk

# Descargar datos necesarios para tokenización de oraciones
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

/home/gabo/Personal/Universidad/04 - Cuarto Año/Tesis/Proyecto/atc-alert-system/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Función NER con Ollama y Contexto

Esta función utiliza el modelo de lenguaje Ollama para realizar Named Entity Recognition (NER) y extracción de relaciones en texto de control de tráfico aéreo, **ahora con contexto de entidades previas**.

**Características:**
- Extrae entidades aeronáuticas (aeródromos, altitudes, aeronaves, etc.)
- Identifica relaciones operativas entre entidades
- Usa un prompt especializado para dominio ATC
- **NUEVO: Mantiene consistencia con entidades extraídas en páginas anteriores**
- Devuelve resultados en formato JSON estructurado

**Parámetros:**
- `texto`: Texto de entrada para analizar
- `entidades_previas`: Lista de entidades extraídas en páginas anteriores (opcional)
- `modelo`: Modelo de Ollama a usar (default: "llama3")

**Mejoras implementadas:**
- Contexto de entidades previas para mantener consistencia
- Nuevo constraint que exige usar texto y etiquetas idénticos para entidades recurrentes
- Deduplicación automática de entidades acumuladas

In [ ]:
def ner_con_ollama(texto, entidades_previas=None, modelo="llama3"):
    """Usa Ollama para NER con contexto de entidades previas"""

    prompt = """
        ATC NER & Relationship Extraction Prompt
        "Extract ALL instances of aeronautical entities and their operational relationships in the following JSON format:

        {
        "entities": [
            {
            "text": "the exact text from the source",
            "label": "the category of the entity",
            "context": "brief semantic description"
            }
        ],
        "relationships": [
            {
            "subject": "Entity A",
            "predicate": "The relationship or action connecting them",
            "object": "Entity B"
            }
        ]
        }

        [Specify Constraints]:

            Focus: Only consider entities and relationships related to Air Traffic Control (ATC) procedures, flight constraints (altitude, speed, heading, position), and facility responsibilities.
            Dynamic Discovery: Do not limit extraction to a fixed list. While you should prioritize standard categories (such as Aerodrome, Altitude, ARTCC, Fix/Position, Sector, and Tower), you must also extract any other domain-specific entity discovered in the text that impacts flight operations or airspace management.
            Relationship Type: Prioritize extracting "subject-predicate-object" triplets that define operational rules or constraints (e.g., "Arrivals - must use - CEDES STAR").
            Entity Consistency: Maintain strict consistency with previously extracted entities. Use identical text and labels for recurring entities. If you encounter an entity that matches a previously extracted one, use the exact same text and label rather than creating variations.
            Relationship Entity Validation: All entities referenced in relationships (both subject and object) MUST appear in the entities section of the current extraction OR be present in the previously extracted entities list. Do not create relationships with entities that have not been explicitly extracted as entities.
    """

    # Añadir entidades previas si existen
    if entidades_previas:
        prompt += "\n\n[Previously Extracted Entities]:\n"
        for entidad in entidades_previas:
            # Soporta entidades como dict incompleto, strings, o cualquier objeto printable
            if isinstance(entidad, dict):
                text = entidad.get("text", "")
                label = entidad.get("label", "Unknown")
            else:
                text = str(entidad)
                label = "Unknown"

            text = (text or "").strip()
            label = (label or "Unknown").strip()
            if not text:
                continue

            prompt += f"- {text} ({label})\n"

    prompt += "\n\n[Input Specification]:\n"
    prompt += (texto or "") + "\nJSON Output ONLY:\n"

    response = ollama.chat(model=modelo, messages=[
        {
            'role': 'user',
            'content': prompt
        },
    ])

    return response['message']['content']

## Extractor de JSON robusto

Esta función extrae JSON válido de texto que puede contener bloques de código o respuestas de LLM.

**Problema resuelto:** JSON no es un lenguaje regular, no se puede parsear con regex debido a anidación infinita.

**Solución:** Parser manual que:
- Busca balance correcto de llaves `{}` considerando strings y escapes
- Maneja bloques de código ```json``` o ```
- Ignora llaves dentro de strings
- Procesa caracteres de escape correctamente
- Soporta anidación arbitraria

**Ventajas sobre regex:**
- Maneja estructuras JSON complejas
- Evita falsos positivos por llaves en strings
- Soporta JSON multilinea correctamente

In [4]:
def extract_json(texto):
    """
    Extrae JSON de un texto usando un parser manual que maneja anidación correctamente.
    """
    
    def encontrar_json_por_llaves(texto):
        """
        Encuentra el JSON válido buscando el balance correcto de llaves {}
        """
        for i, char in enumerate(texto):
            if char == '{':
                # Encontrar el cierre de llave correspondiente
                balance = 1
                j = i + 1
                escape_next = False
                in_string = False
                string_char = None
                
                while j < len(texto) and balance > 0:
                    char_actual = texto[j]
                    
                    if not escape_next:
                        if char_actual == '\\' and in_string:
                            escape_next = True
                        elif char_actual in ['"', "'"] and not escape_next:
                            if not in_string:
                                in_string = True
                                string_char = char_actual
                            elif char_actual == string_char:
                                in_string = False
                                string_char = None
                        elif not in_string:
                            if char_actual == '{':
                                balance += 1
                            elif char_actual == '}':
                                balance -= 1
                    
                    if escape_next:
                        escape_next = False
                    
                    j += 1
                
                if balance == 0:
                    # Encontramos un JSON potencial
                    json_str = texto[i:j]
                    try:
                        return json.loads(json_str), j
                    except json.JSONDecodeError:
                        continue
        
        return None, -1
    
    # Primero intentar encontrar y eliminar bloques de código ```json o ```
    texto_limpio = texto
    patron_bloque = r'```(?:json)?\s*\n(.*?)\s*```'
    match_bloque = re.search(patron_bloque, texto, re.DOTALL)
    
    if match_bloque:
        # Extraer el contenido del bloque
        contenido_bloque = match_bloque.group(1).strip()
        resultado, pos = encontrar_json_por_llaves(contenido_bloque)
        if resultado:
            return resultado
    
    # Si no funciona con bloque, buscar en todo el texto
    resultado, pos = encontrar_json_por_llaves(texto_limpio)
    if resultado:
        return resultado
    
    return None


## Configuración inicial

Definimos las rutas y parámetros para el procesamiento:
- `doc_path`: Ruta al archivo PDF de entrada
- `output_dir`: Directorio base para archivos de salida
- `doc_name`: Nombre del documento derivado del filename

Esta configuración permite procesar diferentes documentos cambiando solo las variables iniciales.

In [5]:
def configure_output(doc_path, output_dir, model_name):
    # Configuración
    doc_name = doc_path.split("/")[-1].split(".")[0] + f"({model_name})"

    # Crear directorio de salida si no existe
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Creado directorio: {output_dir}")

    # Crear carpeta para el documento si no existe
    doc_dir = os.path.join(output_dir, doc_name)
    if not os.path.exists(doc_dir):
        os.makedirs(doc_dir)
        print(f"Creada carpeta para el documento: {doc_dir}")

    return doc_dir

## Procesamiento principal del PDF con Contexto Acumulado

Esta celda ejecuta el flujo completo de procesamiento **mejorado con contexto**:

1. **Extracción de texto**: Lee cada página del PDF y extrae el texto
2. **NER con Ollama**: Aplica reconocimiento de entidades usando el modelo LLM con contexto de entidades previas
3. **Acumulación de entidades**: Mantiene una lista creciente de entidades únicas encontradas
4. **Deduplicación**: Evita repetir entidades idénticas basadas en texto normalizado
5. **Parseo de JSON**: Extrae el JSON estructurado de la respuesta del LLM
6. **Guardado**: Crea archivos JSON por página con:
   - `texto_original`: Texto extraído del PDF
   - `ner`: Entidades y relaciones extraídas (o texto plano si falla)
   - `contexto_entidades_usadas`: Número de entidades de contexto disponibles

**Mejoras clave:**
- **Consistencia**: El LLM recibe contexto de entidades previas para mantener nomenclatura consistente
- **Deduplicación**: Sistema automático para evitar entidades duplicadas
- **Métricas**: Registro de cuántas entidades de contexto se usaron por página

**Salida:** Archivos `pagina_N.json` en el directorio del documento con información de contexto

**Manejo de errores:** Si el parseo JSON falla, guarda la respuesta original del LLM como fallback.

In [ ]:
def kex(
    doc_path,
    doc_dir,
    model_name,
    embed_model_name="sentence-transformers/all-MiniLM-L6-v2",
    context_top_k=50,
    context_threshold=0.4,
    page_embed_max_chars=4000,
    granularity="page",  # "page" o "sentence"
    start_page=1,  # página de inicio (1-indexed)
    initial_context=None,  # nueva: lista de entidades iniciales para bootstrap
):

    def _normalize_entity_text(text):
        return (text or "").lower().strip()

    def _entity_to_embed_text(entity):
        # Usamos text + label + context para dar más señal semántica
        text = entity.get("text", "")
        label = entity.get("label", "Unknown")
        context = entity.get("context", "")
        return f"{text} ({label}) ({context})".strip()

    def _cosine_sim_matrix(a, b):
        # a: (d,), b: (n,d)
        denom = (np.linalg.norm(a) * np.linalg.norm(b, axis=1))
        denom = np.where(denom == 0, 1e-12, denom)
        return (b @ a) / denom

    def select_context_entities(texto_chunk, entities_list, entities_embeddings, top_k, threshold):
        if not entities_list:
            return []

        texto = (texto_chunk or "")
        if page_embed_max_chars and len(texto) > page_embed_max_chars:
            texto = texto[:page_embed_max_chars]

        page_emb = embed_model.encode(texto, normalize_embeddings=False)
        sims = _cosine_sim_matrix(page_emb, entities_embeddings)

        idx_sorted = np.argsort(-sims)
        selected = []
        for idx in idx_sorted[:top_k]:
            sim = float(sims[idx])
            # if sim < threshold:
            #     break
            selected.append(entities_list[idx])

        return selected

    def _process_text_chunks(chunks, entities_by_norm, entity_order_norm, entity_embeddings):
        """Procesa una lista de chunks (páginas u oraciones) y acumula entidades/embeddings."""
        for chunk_text, page_num, chunk_idx in chunks:
            # Seleccionar contexto por embeddings (antes de llamar al LLM)
            entities_list_for_context = [entities_by_norm[n] for n in entity_order_norm]
            if entity_embeddings is None:
                context_entities = []
            else:
                context_entities = select_context_entities(
                    chunk_text,
                    entities_list_for_context,
                    entity_embeddings,
                    context_top_k,
                    context_threshold,
                )

            # Aplicar NER usando Ollama con contexto seleccionado
            resultado_ner = ner_con_ollama(chunk_text, context_entities, model_name)

            # Extraer JSON del resultado
            ner_json = extract_json(resultado_ner)

            if ner_json:
                print(f"Chunk {page_num}-{chunk_idx}: JSON extraído correctamente")

                # Acumular nuevas entidades (evitando duplicados)
                if "entities" in ner_json and isinstance(ner_json["entities"], list):
                    new_entities = []
                    for nueva_entidad in ner_json["entities"]:
                        if not isinstance(nueva_entidad, dict):
                            continue
                        norm = _normalize_entity_text(nueva_entidad.get("text"))
                        if not norm:
                            continue

                        if norm not in entities_by_norm:
                            entities_by_norm[norm] = nueva_entidad
                            entity_order_norm.append(norm)
                            new_entities.append(nueva_entidad)

                    # Actualizar embeddings solo para las nuevas entidades
                    if new_entities:
                        new_emb_texts = [_entity_to_embed_text(e) for e in new_entities]
                        new_embs = embed_model.encode(new_emb_texts, normalize_embeddings=False)
                        new_embs = np.atleast_2d(new_embs)

                        if entity_embeddings is None:
                            entity_embeddings = new_embs
                        else:
                            entity_embeddings = np.vstack([entity_embeddings, new_embs])

                print(f"  - Entidades acumuladas: {len(entities_by_norm)}")
                print(f"  - Entidades en contexto (seleccionadas): {len(context_entities)}")
            else:
                print(f"Chunk {page_num}-{chunk_idx}: No se pudo extraer JSON")

            yield {
                "chunk_text": chunk_text,
                "ner": ner_json if ner_json else resultado_ner,
                "context": {
                    "embedding_model": embed_model_name,
                    "top_k": context_top_k,
                    "threshold": context_threshold,
                    "page_embed_max_chars": page_embed_max_chars,
                    "contexto_entidades_usadas": len(context_entities),
                    "contexto_entidades_seleccionadas": [
                        {"text": e.get("text"), "label": e.get("label"), "context": e.get("context", "")} for e in context_entities
                    ],
                    "entidades_acumuladas_total": len(entities_by_norm),
                },
            }

    # Cargar el documento PDF
    reader = PdfReader(doc_path)
    numero_paginas = len(reader.pages)

    # Validar start_page
    if start_page < 1 or start_page > numero_paginas:
        raise ValueError(f"start_page debe estar entre 1 y {numero_paginas}")

    print(f"Procesando páginas {start_page} a {numero_paginas} ({numero_paginas - start_page + 1} páginas) con granularity='{granularity}'...")

    # Embedding model
    embed_model = SentenceTransformer(embed_model_name)

    # Estructuras acumuladas (inicializadas con initial_context si existe)
    entities_by_norm = {}
    entity_order_norm = []
    entity_embeddings = None

    # Procesar contexto inicial si se proporciona
    if initial_context:
        print(f"Cargando contexto inicial con {len(initial_context)} entidades...")
        for entidad in initial_context:
            if not isinstance(entidad, dict):
                continue
            norm = _normalize_entity_text(entidad.get("text"))
            if not norm:
                continue
            if norm not in entities_by_norm:
                entities_by_norm[norm] = entidad
                entity_order_norm.append(norm)

        # Generar embeddings para el contexto inicial
        if entities_by_norm:
            initial_emb_texts = [_entity_to_embed_text(e) for e in entities_by_norm.values()]
            entity_embeddings = embed_model.encode(initial_emb_texts, normalize_embeddings=False)
            print(f"Contexto inicial procesado: {len(entities_by_norm)} entidades con embeddings")

    # Procesar cada página y crear JSON
    for i in range(start_page - 1, numero_paginas):  # 0-indexed range
        pagina = reader.pages[i]
        texto_original = pagina.extract_text() or ""

        if granularity == "sentence":
            # Dividir en oraciones
            sentences = nltk.sent_tokenize(texto_original)
            chunks = [(sent, i + 1, idx) for idx, sent in enumerate(sentences)]
        else:  # "page"
            chunks = [(texto_original, i + 1, 0)]

        # Procesar chunks de la página
        page_results = list(_process_text_chunks(chunks, entities_by_norm, entity_order_norm, entity_embeddings))

        # Guardar como archivo JSON
        pagina_data = {
            "texto_original": texto_original,
            "granularity": granularity,
            "sentence_results" if granularity == "sentence" else "page_results": page_results,
        }

        filename = f"pagina_{i+1}.json"
        filepath = os.path.join(doc_dir, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(pagina_data, f, ensure_ascii=False, indent=2)

        print(f"Creado archivo: {filename}")

    print(f"\nProceso completado. Archivos guardados en: {doc_dir}")
    print(f"Total de entidades únicas acumuladas: {len(entities_by_norm)}")

In [1]:
def extract_accumulated_entities_from_folder(folder_path):
    """
    Lee todos los JSON de una carpeta y extrae todas las entidades acumuladas
    (útil para exportar catálogo consolidado o usar como initial_context)
    """
    from pathlib import Path
    import json
    
    folder = Path(folder_path)
    if not folder.is_dir():
        raise ValueError(f"La carpeta no existe: {folder_path}")
    
    accumulated = {}  # normalized_text -> entity_info
    
    for json_file in sorted(folder.glob("pagina_*.json")):
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Detectar si es sentence-level o page-level
            chunks = data.get('sentence_results') or [data]  # lista de chunks
            
            for chunk in chunks:
                ner_data = chunk.get('ner', {})
                if not isinstance(ner_data, dict) or 'entities' not in ner_data:
                    continue
                
                for entity in ner_data['entities']:
                    if not isinstance(entity, dict) or 'text' not in entity:
                        continue
                    
                    text = entity['text']
                    normalized = (text or "").lower().strip()
                    if not normalized:
                        continue
                    
                    # Conservar la primera aparición (más completa)
                    if normalized not in accumulated:
                        accumulated[normalized] = {
                            'text': text,
                            'label': entity.get('label', 'Unknown'),
                            'context': entity.get('context', ''),
                            'first_seen_page': data.get('page_number', '?'),
                            'source_file': data.get('source_file', json_file.name),
                        }
                    else:
                        # Podríamos actualizar frecuencia o mantener la primera aparición
                        pass
                        
        except Exception as e:
            print(f"Error procesando {json_file.name}: {e}")
    
    print(f"Entidades únicas acumuladas desde {folder}: {len(accumulated)}")
    return accumulated

In [ ]:
# Ejemplo: extraer entidades acumuladas de una carpeta y usarlas como initial_context
folder_path = "output/4_kex_output/ICAO Standard Phraseology(deepseek-r1:7b)"
try:
    accumulated_entities = extract_accumulated_entities_from_folder(folder_path)
    
    # Mostrar resumen
    print("\n=== RESUMEN DE ENTIDADES ACUMULADAS ===")
    label_counts = {}
    for entity in accumulated_entities.values():
        label = entity['label']
        label_counts[label] = label_counts.get(label, 0) + 1
    
    print(f"Total entidades únicas: {len(accumulated_entities)}")
    print("Distribución por tipo:")
    for label, count in sorted(label_counts.items()):
        print(f"  - {label}: {count}")
    
    # Convertir a lista para usar como initial_context
    initial_context_list = list(accumulated_entities.values())
    
    print(f"\nEntidades listas para usar como initial_context: {len(initial_context_list)}")
    print("Ejemplo de primeras 3 entidades:")
    for i, ent in enumerate(initial_context_list[:3], 1):
        print(f"  {i}. {ent['text']} ({ent['label']}) - {ent['context'][:50]}...")
    
    # Opcional: guardar catálogo completo
    # with open("catalogo_entidades.json", "w", encoding="utf-8") as f:
    #     json.dump(accumulated_entities, f, ensure_ascii=False, indent=2)
    # print("\nCatálogo guardado en: catalogo_entidades.json")
    
    # Ejemplo de cómo usarlo en kex:
    # kex(doc_path, doc_dir, model_name, initial_context=initial_context_list)
    
except Exception as e:
    print(f"Error: {e}")

Entidades únicas acumuladas desde output/4_kex_output/ICAO Standard Phraseology(deepseek-r1:7b) : 330

=== RESUMEN DE ENTIDADES ACUMULADAS ===
Total entidades únicas: 330
Distribución por tipo:
  -  Facility Responsibilities: 1
  - AGC Safety Initiative: 1
  - ARTCC: 1
  - ATC Controller: 1
  - ATC Facility: 1
  - ATC Measures: 1
  - ATC Operations: 1
  - ATC Tower: 1
  - ATC facility: 1
  - Access Control: 1
  - Action: 10
  - Aerodrome: 6
  - Aerodrome Fix: 1
  - Aerodrome Safety: 1
  - Aeronautical: 2
  - Aeronautical Entity: 8
  - Air Traffic Control: 1
  - Aircraft: 5
  - Airport: 2
  - Altitude: 18
  - Approach: 1
  - At: 1
  - Call Sign: 1
  - Callsign: 5
  - Cancel Take-off Command: 1
  - Category: 1
  - Ceiling: 1
  - Clearance: 2
  - Cockpit Workload: 1
  - Communication: 2
  - Communication error (Incident): 1
  - Conditional clearance: 1
  - Conditions: 1
  - Constraint: 23
  - Control Area: 1
  - Controller: 1
  - Crew: 12
  - Departure: 1
  - Descending: 1
  - Descending 

In [ ]:
# Configuración
doc_path = "../docs/ICAO Standard Phraseology.pdf"
output_dir = "kex_output"

doc_dir = configure_output(doc_path, output_dir, "llama3.1")
kex(doc_path, doc_dir, "llama3.1")

---
---

## Reprocesamiento de archivos existentes

Esta celda actualiza los archivos JSON existentes para mejorar la extracción del NER.

**Propósito:**
- Lee archivos JSON ya generados que contienen `llm_output` crudo
- Aplica la función `extract_json()` mejorada para parsear correctamente el NER
- Reemplaza el campo `ner` con el JSON estructurado

**Manejo de errores robusto:**
- Verifica archivos vacíos antes de procesar
- Captura `JSONDecodeError` para archivos corruptos
- Maneja excepciones generales
- Reporta estado de cada archivo procesado

**Resultado:** Archivos JSON con campo `ner` correctamente estructurado en lugar de texto plano.

In [11]:
from pathlib import Path

p = Path("kex_output/ICAO Standard Phraseology")

for file in p.iterdir():
    file_path = "/".join(file.parts)
    
    try:
        # Leer el archivo primero
        with open(file_path, 'r', encoding='utf-8') as f:
            pag_lines = "".join(f.readlines())
            
            # Verificar que el archivo no esté vacío
            if not pag_lines.strip():
                print(f"Archivo vacío: {file.name}")
                continue
                
            pag_json = json.loads(pag_lines)
        
        # Procesar el NER solo si existe llm_output
        if "llm_output" in pag_json:
            pag_json["ner"] = extract_json(pag_json["llm_output"])
        else:
            print(f"No hay llm_output en: {file.name}")
        
        # Escribir el archivo actualizado
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(json.dumps(pag_json, indent=4))
        
        print(f"Procesado: {file.name}")
        
    except json.JSONDecodeError as e:
        print(f"Error JSON en {file.name}: {e}")
    except Exception as e:
        print(f"Error procesando {file.name}: {e}")

Archivo vacío: pagina_6.json
Procesado: pagina_5.json
Procesado: pagina_11.json
Procesado: pagina_12.json
Procesado: pagina_3.json
Procesado: pagina_16.json
Procesado: pagina_17.json
Procesado: pagina_13.json
Procesado: pagina_14.json
Procesado: pagina_8.json
Procesado: pagina_2.json
Procesado: pagina_4.json
Procesado: pagina_15.json
Procesado: pagina_1.json
Procesado: pagina_20.json
Procesado: pagina_10.json
Procesado: pagina_19.json
Procesado: pagina_7.json
Procesado: pagina_18.json
Procesado: pagina_9.json


In [18]:
from pathlib import Path
import pandas as pd
import json

df = pd.DataFrame(columns=["model"] + ["pagina_" + str(i) for i in range(1,21)])
p = Path("kex_output")

for folder in p.iterdir():
    ner_pages = {}
    for file in folder.iterdir():
        key = int(file.stem.split("_")[-1])
        file_path = "/".join(file.parts)
        try:
            # Leer el archivo primero
            with open(file_path, 'r', encoding='utf-8') as f:
                pag_lines = "".join(f.readlines())
                
                # Verificar que el archivo no esté vacío
                if not pag_lines.strip():
                    print(f"Archivo vacío: {file.name}")
                    ner_pages[key] = ""
                    continue
                    
                pag_json = json.loads(pag_lines)
            
            ner_pages[key] = pag_json["ner"]    
        except json.JSONDecodeError as e:
            print(f"Error JSON en {file.name}: {e}")
        except Exception as e:
            print(f"Error procesando {file.name}: {e}")

    ner_pages_sort = []
    for i in sorted(ner_pages):
        ner_pages_sort.append(ner_pages[i])
    
    df.loc[len(df)] = [folder.stem] + ner_pages_sort

Archivo vacío: pagina_6.json
